# Lab MLOps — Analyse de sentiment avec DistilBERT (NLP)

**Objectif :** rejouer le workflow MLOps du ViT, mais sur du **texte**. On fine-tune DistilBERT pour l'analyse de sentiment, on l'évalue, puis on le **publie sur le Hub**. Ensuite (hors notebook) : un **Space Gradio**.

**Dataset :** [`rotten_tomatoes`](https://huggingface.co/datasets/rotten_tomatoes) — critiques de films, 2 classes (0 = négatif, 1 = positif).
**Modèle de base :** `distilbert-base-uncased`.
**Environnement :** Colab avec **GPU** (`Runtime > Change runtime type > T4 GPU`).

> Le seul changement par rapport au ViT : dataset texte, `AutoTokenizer`, `DataCollatorWithPadding`, `AutoModelForSequenceClassification`. Le reste (Trainer, evaluate, push_to_hub) est identique.


## 1. Installation & vérification GPU

In [ ]:
!pip install -q -U transformers datasets evaluate accelerate
import torch
print("PyTorch:", torch.__version__, "| GPU:", torch.cuda.is_available())

## 2. Connexion au Hub (token *Write*)

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## 3. Chargement du dataset

In [ ]:
from datasets import load_dataset

dataset = load_dataset("rotten_tomatoes")
print(dataset)
print(dataset["train"][0])   # {'text': ..., 'label': 0/1}

id2label = {0: "NEGATIVE", 1: "POSITIVE"}
label2id = {"NEGATIVE": 0, "POSITIVE": 1}

## 4. Tokenisation

In [ ]:
from transformers import AutoTokenizer

checkpoint = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

def preprocess(batch):
    return tokenizer(batch["text"], truncation=True, max_length=128)

tokenized = dataset.map(preprocess, batched=True)

## 5. Collator et métrique

In [ ]:
import numpy as np, evaluate
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)  # padding dynamique
accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return accuracy.compute(predictions=predictions, references=labels)

## 6. Modèle

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    checkpoint, num_labels=2, id2label=id2label, label2id=label2id
)

## 7. Entraînement

> API récente : `eval_strategy` (pas `evaluation_strategy`), `Trainer(processing_class=tokenizer)` (pas `tokenizer=`).

In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="distilbert-sentiment-demo",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    report_to="none",
    push_to_hub=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

## 8. Évaluation sur le test

In [ ]:
print(trainer.evaluate(tokenized["test"]))

## 9. Publication sur le Hub

In [ ]:
trainer.push_to_hub()
print("Modèle publié.")

## 10. Test de l'inférence

In [ ]:
from transformers import pipeline
from huggingface_hub import whoami

repo_id = f'{whoami()["name"]}/distilbert-sentiment-demo'
clf = pipeline("sentiment-analysis", model=repo_id)
print(clf("This movie was absolutely fantastic!"))
print(clf("A total waste of time, I fell asleep."))

## ✅ Étape suivante

1. Créez un **Space Gradio** (dossier `space_nlp/`) pour ce modèle.
2. Puis passez au notebook **`model_versioning_colab.ipynb`** pour apprendre à gérer plusieurs versions du modèle.
